In [1]:
import os
import json
import torch
import faiss
import numpy as np
import pandas as pd

from PIL import Image
import open_clip


device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)


annotation_file = "driving_dataset_2k/_annotations.coco.json"

with open(annotation_file, "r") as f:
    coco = json.load(f)


class_mapping = {
    c["id"]: c["name"]
    for c in coco["categories"]
}

print(class_mapping)


clip_model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

clip_model = clip_model.to(device)
clip_model.eval()


index = faiss.read_index(
    "clip_store/driving_clip.index"
)

metadata = pd.read_csv(
    "clip_store/image_metadata.csv"
)

valid_paths = metadata["image_path"].tolist()

print("Images indexed:", index.ntotal)

/home/sanjnasuresh_umass_edu/.conda/envs/driving-clip/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
{0: 'obstacles', 1: 'biker', 2: 'car', 3: 'pedestrian', 4: 'trafficLight', 5: 'trafficLight-Green', 6: 'trafficLight-GreenLeft', 7: 'trafficLight-Red', 8: 'trafficLight-RedLeft', 9: 'trafficLight-Yellow', 10: 'trafficLight-YellowLeft', 11: 'truck'}
Images indexed: 10


In [2]:
@torch.no_grad()
def encode_text(text):

    tokens = tokenizer([text]).to(device)

    features = clip_model.encode_text(tokens)

    features = features / features.norm(
        dim=-1,
        keepdim=True
    )

    return features.cpu().numpy().astype("float32")

In [3]:
def retrieve_images(query, k=10):

    query_embedding = encode_text(query)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "image_path": valid_paths[idx],
            "clip_score": float(score)
        })

    return results

In [4]:
import keras
import keras_cv
import tensorflow as tf

I0000 00:00:1788415264.970191 2908894 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788415271.095260 2908894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
NUM_CLASSES = len(coco["categories"])

In [6]:
detector = keras_cv.models.RetinaNet.from_preset(
    "resnet50_imagenet",
    num_classes=NUM_CLASSES,
    bounding_box_format="xywh"
)

W0000 00:00:1788415283.523757 2908894 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
detector.load_weights(
    "retinanet_resnet50.weights.h5"
)

In [8]:
def prepare_detection_image(image_path):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.cast(
        image,
        tf.float32
    )

    original_shape = tf.shape(image)

    image = tf.image.resize(
        image,
        (640, 640)
    )

    return image, original_shape

In [16]:
def detect_objects(
    image_path,
    confidence_threshold=0.5
):

    image, original_shape = prepare_detection_image(
        image_path
    )

    batch = tf.expand_dims(
        image,
        axis=0
    )

    # Important: pass tensor directly
    predictions = detector.predict(
        batch,
        verbose=0
    )

    print(predictions.keys())

    boxes = predictions["boxes"][0]
    classes = predictions["classes"][0]
    confidence = predictions["confidence"][0]

    detections = []

    for box, cls, score in zip(
        boxes,
        classes,
        confidence
    ):

        score = float(score)

        if score < confidence_threshold:
            continue

        detections.append({
            "class_id": int(cls),
            "confidence": score,
            "bbox": box.tolist()
        })

    return detections

In [17]:
def retrieve_and_detect(
    query,
    retrieval_k=8,
    confidence_threshold=0.5
):

    # ------------------------------
    # Stage 1: CLIP retrieval
    # ------------------------------

    candidates = retrieve_images(
        query,
        k=retrieval_k
    )

    final_results = []

    # ------------------------------
    # Stage 2: object detection
    # ------------------------------

    for candidate in candidates:

        detections = detect_objects(
            candidate["image_path"],
            confidence_threshold=confidence_threshold
        )

        final_results.append({
            "image_path": candidate["image_path"],
            "clip_score": candidate["clip_score"],
            "detections": detections
        })

    return final_results

In [18]:
results = retrieve_and_detect(
    "a pedestrian crossing the road",
    retrieval_k=5
)

dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])


In [19]:
for i, result in enumerate(results):
    print(f"\nIMAGE {i+1}")
    print("Path:", result["image_path"])
    print("CLIP score:", result["clip_score"])
    print("Detections:", result["detections"][:5])


IMAGE 1
Path: driving_dataset_2k/1478732320379283754_jpg.rf.XAb6FgGQlikWLf7uaJlc.jpg
CLIP score: 0.2847323715686798
Detections: []

IMAGE 2
Path: driving_dataset_2k/1478897939548344386_jpg.rf.65f63f2e09ecc105a828fc5243022dd5.jpg
CLIP score: 0.28127986192703247
Detections: []

IMAGE 3
Path: driving_dataset_2k/1478021633083910147_jpg.rf.15f93feaaf39dc81a4ea2dff2e84733a.jpg
CLIP score: 0.2634689211845398
Detections: []

IMAGE 4
Path: driving_dataset_2k/1478900984522923814_jpg.rf.957e827360280e683d88aa41961d7267.jpg
CLIP score: 0.26047804951667786
Detections: []

IMAGE 5
Path: driving_dataset_2k/1478898052093813142_jpg.rf.9HmSxKoa75wFj3EuoCKs.jpg
CLIP score: 0.2534090280532837
Detections: []


In [20]:
target_class = "pedestrian"

In [21]:
def filter_by_object(
    results,
    target_class
):

    filtered = []

    target_class = target_class.lower()

    for result in results:

        matching = [
            d
            for d in result["detections"]
            if d["class_name"].lower() == target_class
        ]

        if len(matching) == 0:
            continue

        best_detection = max(
            matching,
            key=lambda x: x["confidence"]
        )

        result["best_detection"] = best_detection

        filtered.append(result)

    return filtered

In [23]:
results = retrieve_and_detect(
    "a pedestrian crossing the road",
    retrieval_k=5
)

results = filter_by_object(
    results,
    target_class="pedestrian"
)

dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])


In [24]:
def rerank_results(
    results,
    alpha=0.7,
    beta=0.3
):

    for result in results:

        clip_score = result["clip_score"]

        detection_score = (
            result["best_detection"]["confidence"]
        )

        result["final_score"] = (
            alpha * clip_score
            +
            beta * detection_score
        )

    return sorted(
        results,
        key=lambda x: x["final_score"],
        reverse=True
    )

In [25]:
results = rerank_results(results)

In [26]:
def search(
    query,
    target_class,
    retrieval_k=20,
    final_k=5,
    confidence_threshold=0.5
):

    # 1. Semantic retrieval
    results = retrieve_and_detect(
        query,
        retrieval_k=retrieval_k,
        confidence_threshold=confidence_threshold
    )

    # 2. Verify object
    results = filter_by_object(
        results,
        target_class
    )

    # 3. Rerank
    results = rerank_results(
        results
    )

    # 4. Return final Top-K
    return results[:final_k]

In [28]:
results = search(
    query="a pedestrian crossing the road",
    target_class="pedestrian",
    retrieval_k=10,
    final_k=5
)

dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])


In [29]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def show_final_results(results):

    n = len(results)

    if n == 0:
        print("No matching images found.")
        return

    fig, axes = plt.subplots(
        1,
        n,
        figsize=(5 * n, 5)
    )

    if n == 1:
        axes = [axes]

    for ax, result in zip(axes, results):

        image = Image.open(
            result["image_path"]
        ).convert("RGB")

        ax.imshow(image)

        detection = result["best_detection"]

        x, y, w, h = detection["bbox"]

        rect = patches.Rectangle(
            (x, y),
            w,
            h,
            linewidth=2,
            fill=False
        )

        ax.add_patch(rect)

        ax.text(
            x,
            y,
            f'{detection["class_name"]} '
            f'{detection["confidence"]:.2f}'
        )

        ax.set_title(
            f'CLIP: {result["clip_score"]:.3f}\n'
            f'Final: {result["final_score"]:.3f}'
        )

        ax.axis("off")

    plt.show()

In [30]:
results = search(
    query="a pedestrian crossing the road",
    target_class="pedestrian",
    retrieval_k=10,
    final_k=5
)

show_final_results(results)

dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
dict_keys(['boxes', 'confidence', 'classes', 'num_detections'])
No matching images found.
